# Núcleo Fase 3

Este notebook actúa como orquestador de la Fase 3.

La Fase 2 se mantiene congelada como línea base. Todo el código nuevo de
benchmarking vive en F3 y las mediciones de procesamiento/integración se
ejecutan sobre copias temporales.

## 1. Configuración del entorno

In [1]:
from pathlib import Path
import sys

import pandas as pd


def encontrar_raiz_proyecto() -> Path:
    candidatos = [Path.cwd(), *Path.cwd().parents]

    for candidato in candidatos:
        if (candidato / "F2").exists() and (candidato / "F3").exists():
            return candidato

    raise RuntimeError(
        "No se encontró la raíz del repositorio. "
        "Ejecuta el notebook dentro del proyecto."
    )


RAIZ_PROYECTO = encontrar_raiz_proyecto()

if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

print(f"Raíz del proyecto: {RAIZ_PROYECTO}")

Raíz del proyecto: /home/sebastian/Escritorio/Copia de magister/proyectos/grupo_1_programacion_ciencia_datos


In [2]:
from F3.src.nucleo.baseline_f2 import (
    construir_baseline_f2,
    cargar_interim,
    cargar_processed,
)
from F3.src.nucleo.metricas import benchmark
from F3.src.nucleo.rendimiento_f2 import (
    huella_f2,
    medir_extraccion_f2,
    medir_procesamiento_f2,
    medir_integracion_f2,
    resumir_pipeline_f2,
)

## 2. Línea base reproducible de F2

In [3]:
baseline = construir_baseline_f2()
print("Baseline F2 construido correctamente.")

Baseline F2 construido correctamente.


### 2.1 Entorno de ejecución

In [4]:
pd.DataFrame(
    baseline["entorno"].items(),
    columns=["propiedad", "valor"],
)

,propiedad,valor
0,commit_git,da93bc67fe98d2cc741459770acac0ca2af60581
1,python,3.12.14
2,pandas,3.0.5
3,sistema,Linux
4,version_sistema,7.2.6-1-cachyos
5,arquitectura,x86_64


### 2.2 Resumen de datos interim

In [5]:
baseline["resumen_interim"]

,tabla,filas,columnas,nulos,duplicados_exactos,memoria_mb
0,proyecto_ley,15,15,1,0,0.013889
1,diputados,157,13,628,0,0.065296
2,militancias,407,6,0,0,0.128488
3,periodos,11,4,0,0,0.002151
4,detalle_votaciones,1993,20,1993,0,1.372777


### 2.3 Resumen de datos processed

In [6]:
baseline["resumen_processed"]

,tabla,filas,columnas,nulos,duplicados_exactos,memoria_mb
0,diputados_procesados,157,11,157,0,0.060505
1,militancias_observadas,407,8,407,0,0.131981
2,militancias_analiticas,399,10,399,0,0.183864
3,detalle_votaciones_procesado,1993,19,0,0,1.357572
4,proyecto_ley_procesado,15,15,1,0,0.013889
5,reporte_calidad,173,5,0,2,0.099904
6,diagnostico_integracion,15,2,0,0,0.002319
7,big_table_analitica,1993,34,2115,0,3.124971


### 2.4 Inventario de archivos raw

In [7]:
baseline["inventario_raw"]

,etapa,nombre,ruta,tamano_mb,sha256
0,raw,proyecto_ley_xml,F2/data/raw/VotacionesPorProyectoDeLey/proyect...,0.014896,1376e652f72b49db7dbda5362b16e99896b5b698575007...
1,raw,diputados_periodo_10_xml,F2/data/raw/diputados/diputados_periodo_10.xml,0.229163,7df872c435e383d78aaabdfa384f41a7d026fa356a469e...
2,raw,votacion_20627,F2/data/raw/votaciones/votacion_20627.xml,0.032712,a37519cdd0cbb792bacbab08893e4cad013f9f3e5efbec...
3,raw,votacion_20628,F2/data/raw/votaciones/votacion_20628.xml,0.034303,d5bb085bbf733c82d3ca6da433e3c352663d02abb25fb7...
4,raw,votacion_20629,F2/data/raw/votaciones/votacion_20629.xml,0.036100,11b0c85ef243301d2f051b712dfc084fe31a163d142e7d...
5,raw,votacion_20630,F2/data/raw/votaciones/votacion_20630.xml,0.036104,17a515b741c844b8273f250814069bb93fed2b8f277ce6...
6,raw,votacion_20631,F2/data/raw/votaciones/votacion_20631.xml,0.036098,a6a1295797cdc312809098fcec0a31cfcf2bd458cb6054...
7,raw,votacion_20632,F2/data/raw/votaciones/votacion_20632.xml,0.036098,d25e7f4645e2df86c7aa8324ea0e8b92add5453455b788...
8,raw,votacion_20633,F2/data/raw/votaciones/votacion_20633.xml,0.036102,c8a008bcfa8a91c219730b43bc42a316200961182fb93d...
9,raw,votacion_20634,F2/data/raw/votaciones/votacion_20634.xml,0.036100,f4e1138f1d146c13c759d08dde9b2b4596b8410bfc5f69...


### 2.5 Inventario de archivos interim

In [8]:
baseline["inventario_interim"]

,etapa,nombre,ruta,tamano_mb,sha256
0,interim,proyecto_ley,F2/data/interim/VotacionesPorProyectoDeLey/pro...,0.004906,defea1dbb79ae75a2face5e486da29b7089c43f9887ac0...
1,interim,diputados,F2/data/interim/diputados.csv,0.013440,6e848c5c42fdcef259395b7128ad29a7348bfa7f420741...
2,interim,militancias,F2/data/interim/militancias.csv,0.029065,ea91217c815b554fbf80c8bfeeedf9880eb5994a9d8733...
3,interim,periodos,F2/data/interim/periodos.csv,0.000496,9dbe971f1c4197e3564c28c8fb95cc8b76bc696586bab8...
4,interim,detalle_votaciones,F2/data/interim/votaciones/detalle_votaciones.csv,0.280595,d6e1e54bcfb82f43c3f37720dce47fd24d09c9098c5043...


### 2.6 Inventario de archivos processed

In [9]:
baseline["inventario_processed"]

,etapa,nombre,ruta,tamano_mb,sha256
0,processed,diputados_procesados,F2/data/processed/diputados_procesados.csv,0.012240,3a2e120803cbc1086a2721a79ae1b010a0763efa1825fa...
1,processed,militancias_observadas,F2/data/processed/militancias_observadas.csv,0.031041,6a14563020c484c20da5c2d2a647105c6261b5d8a3884b...
2,processed,militancias_analiticas,F2/data/processed/militancias_analiticas.csv,0.048360,49492e32f327364d1fb7eed92da871b42644600579a9bf...
3,processed,detalle_votaciones_procesado,F2/data/processed/detalle_votaciones_procesado...,0.278684,c898218a9325a5ba32319fe21876c761a832ae5ad5837b...
4,processed,proyecto_ley_procesado,F2/data/processed/proyecto_ley_procesado.csv,0.004903,bd0d14a0d01178dd84baf25fa75fc619c243e7d52c3347...
5,processed,reporte_calidad,F2/data/processed/reporte_calidad.csv,0.040108,ed85dee7da41e285fcbed20b9086dfd6fefaf0d35c80d9...
6,processed,diagnostico_integracion,F2/data/processed/diagnostico_integracion.csv,0.000565,02b85cbed3e9a0601a4f80e1321df4c9d380a04f013bed...
7,processed,big_table_analitica,F2/data/processed/big_table_analitica.csv,0.873079,5bf21cbcb360da9074db00ac78de4ab53f2de3c061fa99...


### 2.7 Esquema y cardinalidad interim

In [10]:
baseline["columnas_interim"]

,tabla,columna,dtype,nulos,no_nulos,valores_unicos
0,proyecto_ley,numero_boletin,str,0,15,1
1,proyecto_ley,Id,int64,0,15,15
2,proyecto_ley,Descripcion,str,0,15,1
3,proyecto_ley,Fecha,str,0,15,15
4,proyecto_ley,TotalSi,int64,0,15,10
5,proyecto_ley,TotalNo,int64,0,15,10
6,proyecto_ley,TotalAbstencion,int64,0,15,7
7,proyecto_ley,TotalDispensado,int64,0,15,1
8,proyecto_ley,Quorum,str,0,15,2
9,proyecto_ley,Resultado,str,0,15,1


### 2.8 Esquema y cardinalidad processed

In [11]:
baseline["columnas_processed"]

,tabla,columna,dtype,nulos,no_nulos,valores_unicos
0,diputados_procesados,diputado_id,int64,0,157,157
1,diputados_procesados,nombre,str,0,157,119
2,diputados_procesados,apellido_paterno,str,0,157,143
3,diputados_procesados,apellido_materno,str,0,157,132
4,diputados_procesados,fecha_nacimiento,str,0,157,157
...,...,...,...,...,...,...
99,big_table_analitica,fecha_termino_periodo,float64,1993,0,0
100,big_table_analitica,tiene_votos,bool,0,1993,1
101,big_table_analitica,partido_id,str,0,1993,23
102,big_table_analitica,partido_nombre,str,0,1993,23


## 3. Validación inicial de la infraestructura de métricas

Estas mediciones comprueban que `benchmark()` funciona con datos reales.
Todavía no representan el baseline completo de F2.

In [12]:
_, metricas_interim = benchmark(
    cargar_interim,
    repeticiones=5,
    calentamiento=1,
)

_, metricas_processed = benchmark(
    cargar_processed,
    repeticiones=5,
    calentamiento=1,
)

metricas_iniciales = pd.DataFrame(
    [
        {"etapa": "Carga interim", **metricas_interim},
        {"etapa": "Carga processed", **metricas_processed},
    ]
)

metricas_iniciales

,etapa,repeticiones,calentamiento,tiempo_mediana_s,tiempo_min_s,tiempo_max_s,tiempo_std_s,memoria_pico_mediana_mb,memoria_pico_min_mb,memoria_pico_max_mb,memoria_pico_std_mb
0,Carga interim,5,1,0.007646,0.007377,0.007846,0.000175,1.402487,1.402378,1.405148,0.001203
1,Carga processed,5,1,0.015842,0.015471,0.017144,0.000657,1.795754,1.795675,1.799920,0.001868


## 4. Baseline de rendimiento F2

### Principio de seguridad

`F2/` se considera **solo lectura**.

- extracción lee XML/CSV existentes y no llama servicios externos;
- procesamiento crea un `F2/data/interim` temporal y ejecuta allí la lógica
  original de `F2_03_procesamiento_validacion.ipynb`;
- integración crea un `F2/data/processed` temporal y ejecuta allí la lógica
  original de `F2_04_Integración.ipynb`;
- cualquier `to_csv()` original escribe únicamente dentro del sandbox temporal;
- el sandbox se elimina automáticamente al finalizar cada repetición;
- antes y después del benchmark se calcula una huella SHA-256 del árbol F2;
- si un archivo real de F2 cambia, el benchmark lanza un error.

De esta forma se mide la lógica de F2 sin modificar la línea base.

### 4.0 Verificación inicial de la huella de F2

In [13]:
huella_f2_inicial = huella_f2()

print(f"Archivos controlados en F2: {len(huella_f2_inicial)}")
print("Huella inicial registrada. F2 se tratará como solo lectura.")

Archivos controlados en F2: 49
Huella inicial registrada. F2 se tratará como solo lectura.


### 4.1 Extracción

La extracción se mide de forma local y reproducible.

No se incluyen:

- llamadas HTTP/SOAP;
- latencia de red;
- tiempo del servidor de la Cámara;
- escritura de archivos dentro de F2.

Se reutilizan los XML ya descargados y, cuando la implementación original
trabaja sobre objetos Zeep, se construyen fixtures equivalentes a partir de
los datos persistidos de F2.

In [14]:
resultados_extraccion_f2 = medir_extraccion_f2(
    repeticiones=5,
    calentamiento=1,
)

columnas_mostrar = [
    "etapa",
    "repeticiones",
    "tiempo_mediana_s",
    "tiempo_min_s",
    "tiempo_max_s",
    "tiempo_std_s",
    "memoria_pico_mediana_mb",
    "filas_salida",
]

resultados_extraccion_f2[columnas_mostrar]

,etapa,repeticiones,tiempo_mediana_s,tiempo_min_s,tiempo_max_s,tiempo_std_s,memoria_pico_mediana_mb,filas_salida
0,01 proyecto_ley XML -> DataFrame,5,0.001386,0.001349,0.001476,0.000052,0.134616,15
1,02 periodos: transformación local,5,0.000939,0.000898,0.001601,0.000300,0.272676,11
2,03 diputados/militancias: transformación local,5,0.038386,0.038140,0.061646,0.010431,1.550897,564
3,04 detalle: XML locales -> DataFrame,5,0.027948,0.027869,0.028277,0.000197,1.570319,1993
4,TOTAL extracción reproducible sin red,5,0.061012,0.060828,0.087046,0.011631,1.819490,2583


### 4.2 Procesamiento

Se ejecuta la lógica original de `F2_03_procesamiento_validacion.ipynb`,
pero **no sobre la carpeta F2 real**.

Por cada calentamiento/repetición:

1. se crea una carpeta temporal;
2. se copia `F2/data/interim` a `TEMP/F2/data/interim`;
3. se ejecutan las celdas originales;
4. las exportaciones se escriben en `TEMP/F2/data/processed`;
5. al terminar, la carpeta temporal se elimina.

In [15]:
resultados_procesamiento_f2 = medir_procesamiento_f2(
    repeticiones=5,
    calentamiento=1,
)

resultados_procesamiento_f2[columnas_mostrar]

,diputado_id,nombre,apellido_paterno,apellido_materno
51,1031,Juan,Fuenzalida,Cobo
55,1130,Marta,González,Olea
64,1134,Andrés,Jouannet,Valderrama
124,1070,Patricio,Rosas,Barrientos
155,1184,Roberto,Celedón,Fernández
156,1185,Arturo,Barrios,Oteíza


,diputado_id,partido_id,partido_nombre,partido_alias,fecha_inicio,fecha_termino,intervalo_valido,observacion_calidad
380,1180,IND,Independientes,IND,2026-03-11,2026-03-10 23:59:59,False,None


,diputado_id,partido_a,partido_id_a,fecha_inicio_a,fecha_termino_a,partido_b,partido_id_b,fecha_inicio_b,fecha_termino_b,mismo_partido
0,1017,Unión Demócrata Independiente,UDI,2018-03-11,2025-03-18 23:59:59,Independientes,IND,2020-09-29,2022-03-10 23:59:59,False
1,1114,Federación Regionalista Verde Social,FRVS,2022-03-11,2023-06-12 23:59:59,Independientes,IND,2022-06-13,2024-07-02 23:59:59,False


,partido_id,partido_nombre,fecha_inicio,fecha_termino
70,UDI,Unión Demócrata Independiente,2018-03-11,2025-03-18 23:59:59
71,IND,Independientes,2020-09-29,2022-03-10 23:59:59
72,UDI,Unión Demócrata Independiente,2022-03-11,2025-03-18 23:59:59
73,IND,Independientes,2025-03-19,2026-03-10 23:59:59
74,PREP,Partido Republicano,2026-03-11,2030-03-10 23:59:59


,partido_id,partido_nombre,fecha_inicio,fecha_termino
57,FRVS,Federación Regionalista Verde Social,2022-03-11,2023-06-12 23:59:59
58,IND,Independientes,2022-06-13,2024-07-02 23:59:59
59,FA,Frente Amplio,2024-07-03,2026-03-10 23:59:59
60,FA,Frente Amplio,2026-03-11,2030-03-10 23:59:59


...

,partido_id,partido_nombre,fecha_inicio,fecha_termino
57,FRVS,Federación Regionalista Verde Social,2022-03-11,2023-06-12 23:59:59
58,IND,Independientes,2022-06-13,2024-07-02 23:59:59
59,FA,Frente Amplio,2024-07-03,2026-03-10 23:59:59
60,FA,Frente Amplio,2026-03-11,2030-03-10 23:59:59


,votacion_id,diputado_id,fecha,militancia_observada
0,20627,1114,2023-05-08 19:05:22,AMBIGUA(FRVS/IND)
1,20628,1114,2023-05-08 19:06:49,AMBIGUA(FRVS/IND)
2,20629,1114,2023-05-08 19:08:21,AMBIGUA(FRVS/IND)
3,20630,1114,2023-05-08 19:09:23,AMBIGUA(FRVS/IND)
4,20631,1114,2023-05-08 19:10:13,AMBIGUA(FRVS/IND)
5,20632,1114,2023-05-08 19:11:06,AMBIGUA(FRVS/IND)
6,20633,1114,2023-05-08 19:11:56,AMBIGUA(FRVS/IND)
7,20634,1114,2023-05-08 19:12:58,AMBIGUA(FRVS/IND)
8,20635,1114,2023-05-08 19:13:51,AMBIGUA(FRVS/IND)
9,20636,1114,2023-05-08 19:15:04,AMBIGUA(FRVS/IND)


,votacion_id,diputado_id,fecha,militancia_observada
0,20627,1180,2023-05-08 19:05:22,RD
1,20628,1180,2023-05-08 19:06:49,RD
2,20629,1180,2023-05-08 19:08:21,RD
3,20630,1180,2023-05-08 19:09:23,RD
4,20631,1180,2023-05-08 19:10:13,RD
5,20632,1180,2023-05-08 19:11:06,RD
6,20633,1180,2023-05-08 19:11:56,RD
7,20634,1180,2023-05-08 19:12:58,RD
8,20635,1180,2023-05-08 19:13:51,RD
9,20636,1180,2023-05-08 19:15:04,RD


,diputado_id,partido_id,partido_nombre,partido_alias,fecha_inicio,fecha_termino,intervalo_valido,observacion_calidad,fecha_termino_original,regla_asignacion_partido
57,1114,FRVS,Federación Regionalista Verde Social,FRVS,2022-03-11,2026-03-10 23:59:59,True,None,2023-06-12 23:59:59,extension_caso_particular_bugueno
72,1017,UDI,Unión Demócrata Independiente,UDI,2022-03-11,2025-03-18 23:59:59,True,None,2025-03-18 23:59:59,vigencia_temporal_estandar
74,1017,PREP,Partido Republicano,PREP,2026-03-11,2030-03-10 23:59:59,True,None,2030-03-10 23:59:59,vigencia_temporal_estandar
377,1180,RD,Revolución Democrática,RD,2022-03-11,2026-03-10 23:59:59,True,None,2024-05-30 23:59:59,extension_caso_particular_veloso
379,1180,FA,Frente Amplio,FA,2026-03-11,2030-03-10 23:59:59,True,None,2030-03-10 23:59:59,vigencia_temporal_estandar


,etapa,repeticiones,tiempo_mediana_s,tiempo_min_s,tiempo_max_s,tiempo_std_s,memoria_pico_mediana_mb,filas_salida
0,configuración y carga,5,0.016924,0.016280,0.018837,0.000968,1.415204,NaN
1,normalización general,5,0.052985,0.050333,0.062059,0.004560,0.769765,NaN
2,procesamiento diputados,5,0.036843,0.034834,0.042356,0.002809,0.133013,157.0
3,procesamiento militancias,5,0.626988,0.599174,0.680693,0.035285,0.724350,399.0
4,procesamiento detalle votaciones,5,0.011596,0.010276,0.011794,0.000636,0.542868,1993.0
5,procesamiento proyecto ley,5,0.015567,0.014875,0.017626,0.001093,0.122542,15.0
6,exportación a sandbox temporal,5,0.089801,0.087753,0.100455,0.005072,0.644813,173.0
7,TOTAL procesamiento,5,0.864856,0.822981,0.901515,0.034955,1.415204,NaN


### 4.3 Integración

Se ejecuta la lógica original de `F2_04_Integración.ipynb` en otro sandbox.

Por cada repetición:

1. se copia `F2/data/processed` a una carpeta temporal;
2. la integración trabaja exclusivamente con esa copia;
3. `big_table_analitica.csv` y `diagnostico_integracion.csv`, si se vuelven a
   generar, se guardan solo en el sandbox;
4. el sandbox se elimina al terminar.

La etapa `integración temporal militancias` será la referencia principal para
comparar posteriormente las estrategias iterativa y vectorizada de F3.

In [16]:
resultados_integracion_f2 = medir_integracion_f2(
    repeticiones=5,
    calentamiento=1,
)

resultados_integracion_f2[columnas_mostrar]

,diputado_id,nombre,apellido_paterno,apellido_materno,opcion_codigo,opcion_voto,votacion_id,descripcion,fecha,total_si,total_no,total_abstencion,total_dispensado,quorum_codigo,quorum,resultado_codigo,resultado,tipo_codigo,tipo,fila_voto_id
0,803,René,Alinco,Bustos,0,En Contra,42724,Boletín N° 11092-07,2024-08-26 19:03:25,65,22,36,0,2,Quórum Calificado,1,Aprobado,1,Proyecto de Ley,42724_803
1,815,Sergio,Bobadilla,Muñoz,2,Abstención,42724,Boletín N° 11092-07,2024-08-26 19:03:25,65,22,36,0,2,Quórum Calificado,1,Aprobado,1,Proyecto de Ley,42724_815
2,872,Jaime,Mulet,Martínez,1,Afirmativo,42724,Boletín N° 11092-07,2024-08-26 19:03:25,65,22,36,0,2,Quórum Calificado,1,Aprobado,1,Proyecto de Ley,42724_872
3,917,Gastón,Von Mühlenbrock,Zamora,0,En Contra,42724,Boletín N° 11092-07,2024-08-26 19:03:25,65,22,36,0,2,Quórum Calificado,1,Aprobado,1,Proyecto de Ley,42724_917
4,948,Gaspar,Rivas,Sánchez,2,Abstención,42724,Boletín N° 11092-07,2024-08-26 19:03:25,65,22,36,0,2,Quórum Calificado,1,Aprobado,1,Proyecto de Ley,42724_948


,validacion,fuera_catalogo,estado
0,Votos con votación inexistente en proyecto_ley,0,OK
1,Votaciones de proyecto_ley sin detalle,0,OK
2,Votos con diputado inexistente,0,OK
3,Votos de diputados sin ninguna militancia anal...,0,OK


,validacion,resultado,estado,detalle
0,Conteo de filas = detalle original,1993 vs 1993,OK,
1,Clave votacion_id + diputado_id única,0,OK,
2,fila_voto_id única,0,OK,
3,Sin columnas duplicadas,0,OK,
4,Sin columnas con sufijo _x/_y,[],OK,
5,Todos los votos tienen datos de proyecto_ley,0,OK,
6,Todos los votos tienen datos de diputado,0,OK,
7,Todos los votos tienen partido analítico,0,OK,
8,Militancia vigente en la fecha del voto (exact...,0,OK,
9,Totales/metadatos de detalle vs. proyecto_ley ...,0,OK,


,metrica,valor
0,filas totales,1993
1,votaciones únicas,15
2,diputados únicos,151
3,partidos analíticos únicos (partido_alias),23
4,rango de fechas,2023-05-08 19:05:22 → 2024-08-26 19:03:25
5,duplicados de clave (votacion_id + diputado_id),0
6,votos sin datos de diputado,0
7,votos sin datos de proyecto_ley,0
8,votos sin militancia vigente,0
9,votos con más de una militancia vigente,0


## ✅ Integración aprobada

Todas las validaciones de la Sección 9 pasaron. `big_table_analitica.csv` refleja el resultado final.

...

,validacion,fuera_catalogo,estado
0,Votos con votación inexistente en proyecto_ley,0,OK
1,Votaciones de proyecto_ley sin detalle,0,OK
2,Votos con diputado inexistente,0,OK
3,Votos de diputados sin ninguna militancia anal...,0,OK


,validacion,resultado,estado,detalle
0,Conteo de filas = detalle original,1993 vs 1993,OK,
1,Clave votacion_id + diputado_id única,0,OK,
2,fila_voto_id única,0,OK,
3,Sin columnas duplicadas,0,OK,
4,Sin columnas con sufijo _x/_y,[],OK,
5,Todos los votos tienen datos de proyecto_ley,0,OK,
6,Todos los votos tienen datos de diputado,0,OK,
7,Todos los votos tienen partido analítico,0,OK,
8,Militancia vigente en la fecha del voto (exact...,0,OK,
9,Totales/metadatos de detalle vs. proyecto_ley ...,0,OK,


,metrica,valor
0,filas totales,1993
1,votaciones únicas,15
2,diputados únicos,151
3,partidos analíticos únicos (partido_alias),23
4,rango de fechas,2023-05-08 19:05:22 → 2024-08-26 19:03:25
5,duplicados de clave (votacion_id + diputado_id),0
6,votos sin datos de diputado,0
7,votos sin datos de proyecto_ley,0
8,votos sin militancia vigente,0
9,votos con más de una militancia vigente,0


## ✅ Integración aprobada

Todas las validaciones de la Sección 9 pasaron. `big_table_analitica.csv` refleja el resultado final.

,etapa,repeticiones,tiempo_mediana_s,tiempo_min_s,tiempo_max_s,tiempo_std_s,memoria_pico_mediana_mb,filas_salida
0,configuración y carga,5,0.049420,0.048427,0.054942,0.002682,0.799695,NaN
1,validaciones referenciales,5,0.040875,0.039806,0.045704,0.002758,0.184371,NaN
2,integración proyecto_ley,5,0.023104,0.021467,0.024375,0.001203,0.230042,NaN
3,integración diputados,5,0.012559,0.011778,0.013368,0.000666,0.251600,NaN
4,integración temporal militancias,5,3.198322,3.103083,3.585994,0.196226,3.183057,1993.0
5,validación casos especiales,5,0.002463,0.002432,0.002851,0.000176,0.064601,NaN
6,validaciones big table,5,0.017147,0.016995,0.019658,0.001115,0.326878,NaN
7,diagnóstico final,5,0.009208,0.008190,0.010041,0.000892,0.136355,NaN
8,exportación a sandbox temporal,5,0.114575,0.101423,0.117910,0.007789,1.543898,1993.0
9,criterio de término,5,0.001298,0.001238,0.001312,0.000036,0.054893,NaN


### 4.4 Resumen del pipeline F2

Este resumen utiliza los resultados ya medidos. **No vuelve a ejecutar F2**.

El total del pipeline es una suma descriptiva de las medianas de extracción,
procesamiento e integración. No representa una ejecución única end-to-end con
red.

In [17]:
resumen_pipeline_f2 = resumir_pipeline_f2(
    resultados_extraccion_f2,
    resultados_procesamiento_f2,
    resultados_integracion_f2,
)

resumen_pipeline_f2[
    [
        "etapa",
        "repeticiones",
        "tiempo_mediana_s",
        "tiempo_min_s",
        "tiempo_max_s",
        "memoria_pico_mediana_mb",
    ]
]

,etapa,repeticiones,tiempo_mediana_s,tiempo_min_s,tiempo_max_s,memoria_pico_mediana_mb
0,TOTAL extracción reproducible sin red,5,0.061012,0.060828,0.087046,1.81949
1,TOTAL procesamiento,5,0.864856,0.822981,0.901515,1.415204
2,TOTAL integracion,5,3.479183,3.357592,3.863077,3.183057
3,TOTAL pipeline F2 reproducible,5,4.405052,4.241402,4.851638,None


### 4.5 Etapas dominantes

In [18]:
etapas_f2 = pd.concat(
    [
        resultados_extraccion_f2[
            ~resultados_extraccion_f2["etapa"].str.startswith("TOTAL")
        ],
        resultados_procesamiento_f2[
            ~resultados_procesamiento_f2["etapa"].str.startswith("TOTAL")
        ],
        resultados_integracion_f2[
            ~resultados_integracion_f2["etapa"].str.startswith("TOTAL")
        ],
    ],
    ignore_index=True,
)

etapas_f2[
    [
        "tipo",
        "etapa",
        "tiempo_mediana_s",
        "memoria_pico_mediana_mb",
        "filas_salida",
    ]
].sort_values(
    "tiempo_mediana_s",
    ascending=False,
).reset_index(drop=True)

,tipo,etapa,tiempo_mediana_s,memoria_pico_mediana_mb,filas_salida
0,integracion,integración temporal militancias,3.198322,3.183057,1993.0
1,procesamiento,procesamiento militancias,0.626988,0.724350,399.0
2,integracion,exportación a sandbox temporal,0.114575,1.543898,1993.0
3,procesamiento,exportación a sandbox temporal,0.089801,0.644813,173.0
4,procesamiento,normalización general,0.052985,0.769765,NaN
5,integracion,configuración y carga,0.049420,0.799695,NaN
6,integracion,validaciones referenciales,0.040875,0.184371,NaN
7,extraccion,03 diputados/militancias: transformación local,0.038386,1.550897,564.0
8,procesamiento,procesamiento diputados,0.036843,0.133013,157.0
9,extraccion,04 detalle: XML locales -> DataFrame,0.027948,1.570319,1993.0


### 4.6 Verificación final de integridad de F2

In [19]:
huella_f2_final = huella_f2()

assert huella_f2_inicial == huella_f2_final, (
    "F2 cambió durante las mediciones."
)

print("OK: F2 permanece idéntica después de ejecutar toda la sección 4.")

OK: F2 permanece idéntica después de ejecutar toda la sección 4.


### 4.7 Limitaciones metodológicas

- `tracemalloc` mide principalmente memoria administrada por Python; no toda
  la memoria nativa de NumPy/pandas.
- la extracción es un benchmark offline de transformación y no del servicio
  externo;
- procesamiento e integración incluyen la lógica de exportación original,
  pero esas escrituras ocurren solamente en almacenamiento temporal;
- el tiempo total del pipeline es una suma descriptiva de medianas de bloques;
- antes de comparar F2 y F3 se deberá validar equivalencia de resultados.

## 5. Integración de componentes F3

In [ ]:
# Pendiente de integrar los componentes desarrollados por Yerko y José.

## 6. Equivalencia F2 vs F3

In [ ]:
# Pendiente: validar igualdad de resultados antes de comparar rendimiento.

## 7. Benchmark comparativo F2 vs F3

In [ ]:
# Pendiente hasta integrar las variantes F3.

## 8. Conclusiones

Las conclusiones finales deberán distinguir resultados observados, equivalencia
funcional, diferencias de rendimiento, limitaciones de medición y decisiones
de diseño.